# EDA inicial — Lending Club accepted loans

Estudo exploratório do dataset de empréstimos aceitos (2007–2018Q4) para o baseline de credit scoring.

Objetivos:

- Conferir volume, alvo e cobertura temporal antes de treinar o modelo.
- Separar variáveis disponíveis na concessão de variáveis que vazam o desfecho.
- Excluir identificadores e texto livre da análise, em linha com a LGPD.
- Orientar o corte entre dataset de referência e dataset de produção.

Fonte: [wordsforthewise/lending-club](https://www.kaggle.com/datasets/wordsforthewise/lending-club), licença CC0.
Arquivo local, fora do Git: `data/raw/accepted_2007_to_2018Q4.csv.gz`.


## 1. Ambiente e escopo

A leitura percorre o arquivo compactado em blocos. O perfil de status e de ano usa a população inteira. Gráficos de distribuição usam uma amostra reproduzível de 4% dos empréstimos com desfecho fechado.

Não entram nesta análise:

- Identificadores e quase identificadores: `id`, `member_id`, `url`, `emp_title`, `zip_code`, `desc`, `title`.
- Variáveis posteriores à concessão: pagamentos, recuperações, FICO atual, hardship e settlement. Usá-las no modelo vazaria o alvo.


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "accepted_2007_to_2018Q4.csv.gz"
SAMPLE_FRACTION = 0.04
RANDOM_STATE = 42
CLOSED_STATUSES = ("Fully Paid", "Charged Off")

APPLICATION_COLUMNS = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "loan_status",
    "purpose",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "pub_rec_bankruptcies",
]

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print(f"pandas {pd.__version__}")
print(DATA_PATH)
print(f"arquivo encontrado: {DATA_PATH.exists()}")


## 2. Perfil da população e amostra

`Charged Off` é o evento de inadimplência. `Fully Paid` é o não evento. Status em aberto (`Current`, `Late`, `In Grace Period`) ficam de fora do alvo porque o desfecho ainda não ocorreu.

Status antigos no formato `Does not meet the credit policy. Status:...` são normalizados para o desfecho correspondente.


In [ ]:
def normalize_status(status: str) -> str:
    """Reduz o status bruto ao desfecho de crédito, quando ele existe.

    Args:
        status: Valor original de `loan_status`.

    Returns:
        `Fully Paid`, `Charged Off` ou o status original.
    """
    text = str(status)
    if "Fully Paid" in text:
        return "Fully Paid"
    if "Charged Off" in text:
        return "Charged Off"
    return text


def profile_and_sample(path: Path, fraction: float, seed: int) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Percorre o arquivo uma vez e devolve contagens populacionais e a amostra.

    Args:
        path: Caminho do CSV compactado.
        fraction: Fração de empréstimos fechados retirada de cada bloco.
        seed: Semente da amostragem.

    Returns:
        Amostra de empréstimos fechados, contagem de status e contagem por ano.
    """
    status_counts: Counter[str] = Counter()
    year_counts: Counter[int] = Counter()
    parts: list[pd.DataFrame] = []

    reader = pd.read_csv(
        path,
        usecols=APPLICATION_COLUMNS,
        chunksize=250_000,
        low_memory=False,
    )
    for chunk in reader:
        status_counts.update(chunk["loan_status"].astype(str).value_counts().to_dict())
        issued = pd.to_datetime(chunk["issue_d"], format="%b-%Y", errors="coerce")
        year_counts.update(issued.dt.year.dropna().astype(int).value_counts().to_dict())

        chunk = chunk.copy()
        chunk["loan_status"] = chunk["loan_status"].map(normalize_status)
        closed = chunk[chunk["loan_status"].isin(CLOSED_STATUSES)]
        if not closed.empty:
            parts.append(closed.sample(frac=fraction, random_state=seed))

    sample = pd.concat(parts, ignore_index=True)
    return sample, pd.Series(status_counts).sort_values(ascending=False), pd.Series(year_counts).sort_index()


sample, status_counts, year_counts = profile_and_sample(DATA_PATH, SAMPLE_FRACTION, RANDOM_STATE)

print(f"registros no arquivo: {int(status_counts.sum()):,}")
print(f"amostra fechada: {len(sample):,}")
status_counts.to_frame("registros")


In [ ]:
population = pd.DataFrame({"ano": year_counts.index.astype(int), "registros": year_counts.values})
closed_rate = sample["loan_status"].value_counts(normalize=True).rename("proporcao")
target_counts = sample["loan_status"].value_counts().rename("amostra")
pd.concat([target_counts, closed_rate], axis=1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

status_counts.head(8).sort_values().plot.barh(ax=axes[0], color="#4C78A8")
axes[0].set_title("Status na população")
axes[0].set_xlabel("Registros")

axes[1].bar(population["ano"], population["registros"], color="#4C78A8")
axes[1].set_title("Volume de emissões por ano")
axes[1].set_xlabel("Ano de issue_d")
axes[1].set_ylabel("Registros")

fig.tight_layout()


## 3. Preparação da amostra

O alvo do baseline será binário: `default = 1` quando o status normalizado é `Charged Off`.

`issue_d` permite, na etapa seguinte, separar referência e produção por tempo. Um corte natural aparece depois de 2015, quando o volume muda de patamar. A decisão final do corte fica para o pipeline, não para este notebook.


In [ ]:
def prepare_sample(frame: pd.DataFrame) -> pd.DataFrame:
    """Tipa datas e cria o alvo binário da amostra fechada.

    Args:
        frame: Amostra com status já normalizado.

    Returns:
        Cópia com `issue_d` em data e `default` inteiro.
    """
    prepared = frame.copy()
    prepared["issue_d"] = pd.to_datetime(prepared["issue_d"], format="%b-%Y", errors="coerce")
    prepared["issue_year"] = prepared["issue_d"].dt.year
    prepared["default"] = (prepared["loan_status"] == "Charged Off").astype(int)
    prepared["term"] = prepared["term"].astype(str).str.strip()
    return prepared


loans = prepare_sample(sample)
print(f"taxa de default na amostra: {loans['default'].mean():.2%}")
print(f"período: {loans['issue_d'].min():%Y-%m} a {loans['issue_d'].max():%Y-%m}")
loans.head(3)


## 4. Qualidade dos campos de concessão

Estas ausências orientam as regras do data contract. Para a etapa 1, candidatas a regras rígidas:

- `loan_amnt`, `int_rate`, `annual_inc`, `dti` e `fico_range_low` não podem ser nulos.
- `annual_inc` deve ser positivo e `dti` não pode ser negativo.
- `loan_status` precisa pertencer ao domínio conhecido.
- `term` precisa ser `36 months` ou `60 months`.


In [ ]:
missing = (
    loans.isna()
    .mean()
    .mul(100)
    .rename("ausentes_pct")
    .sort_values(ascending=False)
    .to_frame()
)
missing["ausentes"] = loans.isna().sum()
missing.head(12)


In [ ]:
numeric_summary = loans[
    ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_range_low", "revol_util"]
].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
numeric_summary


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_columns = ["loan_amnt", "int_rate", "dti", "fico_range_low"]
for axis, column in zip(axes.ravel(), plot_columns):
    sns.histplot(
        data=loans,
        x=column,
        hue="loan_status",
        bins=40,
        element="step",
        stat="density",
        common_norm=False,
        ax=axis,
    )
    axis.set_title(column)

fig.suptitle("Distribuição por desfecho", y=1.02)
fig.tight_layout()


## 5. Categorias e risco observado

Grade, prazo e finalidade são as primeiras variáveis categóricas do contrato. A taxa abaixo é descritiva: não está ajustada por mix de crédito nem por ano.


In [ ]:
def default_rate_by(frame: pd.DataFrame, column: str, min_count: int = 50) -> pd.DataFrame:
    """Calcula volume e taxa de default por categoria.

    Args:
        frame: Amostra com a coluna `default`.
        column: Categoria a agregar.
        min_count: Frequência mínima para manter a categoria.

    Returns:
        Tabela ordenada pela taxa de default.
    """
    grouped = (
        frame.groupby(column, dropna=False)
        .agg(emprestimos=("default", "size"), defaults=("default", "sum"), taxa_default=("default", "mean"))
        .reset_index()
    )
    return grouped[grouped["emprestimos"] >= min_count].sort_values("taxa_default", ascending=False)


grade_risk = default_rate_by(loans, "grade", min_count=1).sort_values("grade")
purpose_risk = default_rate_by(loans, "purpose")
term_risk = default_rate_by(loans, "term", min_count=1)
grade_risk


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.barplot(data=grade_risk, x="grade", y="taxa_default", color="#E45756", ax=axes[0])
axes[0].set_title("Taxa de default por grade")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

sns.barplot(data=term_risk, x="term", y="taxa_default", color="#F58518", ax=axes[1])
axes[1].set_title("Taxa de default por prazo")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))

fig.tight_layout()


In [ ]:
top_purposes = purpose_risk.sort_values("emprestimos", ascending=False).head(10)
fig, axis = plt.subplots(figsize=(8, 5))
sns.barplot(data=top_purposes, y="purpose", x="taxa_default", color="#54A24B", ax=axis)
axis.set_title("Taxa de default nas finalidades mais frequentes")
axis.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
fig.tight_layout()


## 6. Tempo e associação entre variáveis

A série anual mostra se a taxa de default muda com o ciclo. Isso ainda não é concept drift: o mix de grade, prazo e renda também pode ter mudado. A comparação formal fica para o relatório do Evidently.

O mapa de correlação usa apenas variáveis numéricas de concessão. `installment` tende a acompanhar `loan_amnt`; as duas não devem entrar juntas sem critério.


In [ ]:
yearly = default_rate_by(loans.dropna(subset=["issue_year"]), "issue_year", min_count=30)
yearly["issue_year"] = yearly["issue_year"].astype(int)
yearly = yearly.sort_values("issue_year")

fig, axis = plt.subplots(figsize=(10, 4.5))
axis.plot(yearly["issue_year"], yearly["taxa_default"], marker="o", color="#E45756")
axis.set_title("Taxa de default na amostra, por ano de emissão")
axis.set_xlabel("Ano")
axis.set_ylabel("Taxa de default")
axis.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
fig.tight_layout()
yearly


In [ ]:
correlation_columns = [
    "loan_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_range_low",
    "revol_util",
    "inq_last_6mths",
    "delinq_2yrs",
    "default",
]
correlation = loans[correlation_columns].corr(numeric_only=True)

fig, axis = plt.subplots(figsize=(9, 7))
sns.heatmap(correlation, cmap="vlag", center=0, annot=True, fmt=".2f", ax=axis)
axis.set_title("Correlação na amostra")
fig.tight_layout()


## 7. Leitura para as próximas etapas

- O alvo supervisionado deve usar somente empréstimos `Fully Paid` e `Charged Off`.
- O contrato inicial pode exigir não nulos e domínios fechados em valor, taxa, renda, DTI, FICO, prazo e status.
- `grade` e `int_rate` separam risco já na análise descritiva. São candidatas à simulação de drift, junto com `annual_inc` e `dti`.
- `addr_state` fica restrito a monitoramento de mix. Não deve virar feature de decisão sem avaliação de viés.
- Referência e produção devem ser cortadas por `issue_d`, não por amostra aleatória.
- Identificadores, ZIP, URL, cargo e texto livre não devem ser persistidos no pipeline de treino nem nos logs de predição.

Este notebook não treina o modelo e não mede drift. Ele só fixa o recorte, o alvo e as variáveis que podem entrar na concessão.
